# Fast-Diag Polyp Dataset Retry

Retries the 5 datasets that failed in the first ingestion run:
- cvc-colondb (403 → fixed Kaggle ID)
- bkai-igh (competition → dataset upload)
- etis-larib (403 → fixed Kaggle ID)
- polypdb (OSF 500 → retry)
- ldpolypvideo (GitHub archive → Google Drive)

In [1]:
# Cell 1: Mount Drive + Install shared_config
from google.colab import drive
drive.mount('/content/drive')

# Ensure shared_config is importable
import sys


Mounted at /content/drive
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for shared_config (pyproject.toml) ... done


In [2]:
# Cell 2: Kaggle API setup
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/.secrets/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
# Cell 3: Force reload catalog with fixed entries
import importlib
import shared_config.catalog
importlib.reload(shared_config.catalog)
from shared_config.catalog import download_dataset, DATASETS

# Verify the fixed entries
for ds in ['cvc-colondb', 'bkai-igh', 'etis-larib', 'polypdb', 'ldpolypvideo']:
    info = DATASETS.get(ds, {})
    print(f"{ds}: source={info.get('source')}, id={info.get('kaggle_id', info.get('url', 'N/A'))}")

cvc-colondb: source=kaggle, id=hopmai/cvc-colondb
bkai-igh: source=kaggle, id=phamsohanh/bkai-igh-neopolypsmall
etis-larib: source=kaggle, id=nguyenvoquocduong/etis-laribpolypdb
polypdb: source=url, id=https://osf.io/pr7ms/download
ldpolypvideo: source=url, id=https://drive.google.com/drive/folders/13KwU_uZcxsl6dL-mqcs39Yb0gjU9vn3G


In [4]:
# Cell 4: Download the 3 fixed Kaggle datasets + PolypDB retry
import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/data_lake')

retry_datasets = ['cvc-colondb', 'bkai-igh', 'etis-larib', 'polypdb']
results = {}

for ds in retry_datasets:
    print(f"\n{'='*60}")
    print(f"  RETRYING: {ds}")
    print(f"{'='*60}")
    # Clean up any empty directories from failed attempts
    for subdir in ['01_bronze_medical', '00_landing']:
        failed_dir = DRIVE / subdir / ds
        if failed_dir.exists() and not any(failed_dir.rglob('*.*')):
            shutil.rmtree(failed_dir)
            print(f"  Cleaned up empty dir: {failed_dir}")
    try:
        result = download_dataset(ds)
        results[ds] = 'success' if result else 'skipped'
    except Exception as e:
        results[ds] = f'error: {str(e)[:200]}'
        print(f'ERROR: {e}')

print(f"\n\n{'='*60}")
print('KAGGLE + OSF RETRY SUMMARY')
print(f"{'='*60}")
for ds, status in results.items():
    icon = '\u2705' if status == 'success' else ('\u23ed\ufe0f' if status == 'skipped' else '\u274c')
    print(f'{icon} {ds}: {status}')


  RETRYING: cvc-colondb
  Dataset URL: https://www.kaggle.com/datasets/hopmai/cvc-colondb
  License(s): unknown
✅ cvc-colondb ingested to /content/drive/MyDrive/data_lake/01_bronze_medical/cvc-colondb
📋 MANIFEST.json updated: cvc-colondb (1520 files, 182.5 MB)

  RETRYING: bkai-igh
  ⚠️  Dataset access denied: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset
❌ bkai-igh download failed

  RETRYING: etis-larib
  Dataset URL: https://www.kaggle.com/datasets/nguyenvoquocduong/etis-laribpolypdb
  License(s): unknown
✅ etis-larib ingested to /content/drive/MyDrive/data_lake/01_bronze_medical/etis-larib
📋 MANIFEST.json updated: etis-larib (392 files, 176.7 MB)

  RETRYING: polypdb
  ⚠️  Attempt 1/3 failed: HTTP Error 500: Internal Server Error
     Retrying in 5s...
  ⚠️  Attempt 2/3 failed: HTTP Error 500: Internal Server Error
     Retrying in 10s...
  ❌ All 3 attempts failed: HTTP Error 500: Internal Server Error
❌ polypdb download faile

In [5]:
# Cell 5: LDPolypVideo - Download from Google Drive using gdown
!pip install -q gdown
import os, shutil

target = '/content/drive/MyDrive/data_lake/01_bronze_medical/ldpolypvideo'

# Remove old incomplete download (only annotation files from GitHub)
if os.path.exists(target):
    existing_files = [f for f in os.listdir(target) if not f.startswith('.')]
    total_size = sum(
        os.path.getsize(os.path.join(target, f))
        for f in existing_files
        if os.path.isfile(os.path.join(target, f))
    )
    if total_size < 10_000_000:  # Less than 10MB = incomplete
        print(f'Removing incomplete download ({total_size} bytes)...')
        shutil.rmtree(target)

os.makedirs(target, exist_ok=True)

# Download Part-1 (Google Drive folder with videos)
print('\nDownloading Part-1 (Google Drive folder)...')
!gdown --folder "https://drive.google.com/drive/folders/13KwU_uZcxsl6dL-mqcs39Yb0gjU9vn3G" -O {target} --remaining-ok

# Download Part-2 (Single zip file)
print('\nDownloading Part-2 (Single zip)...')
!gdown "1pxFYO-nRd5uqdYsjs7NRkwXQMMurbWsZ" -O {target}/ldpolypvideo_part2.zip

# Extract Part-2 if it's a zip
part2_zip = os.path.join(target, 'ldpolypvideo_part2.zip')
if os.path.exists(part2_zip):
    import zipfile
    print('Extracting Part-2...')
    with zipfile.ZipFile(part2_zip, 'r') as z:
        z.extractall(target)
    os.remove(part2_zip)
    print('Part-2 extracted and zip removed.')

print('\nLDPolypVideo download complete!')

Removing incomplete download (0 bytes)...

Retrieving folder contents
Processing file 1gSJiUhZJnpkELXoCUgNsPtYnUCS7qqyP Test.rar
Processing file 1Y-fS9vMGvrEQZS1rKDnCxSkvrQz1lyXX TrainValid.rar
Processing file 1vgC5UU3kH8OVdRPfBxIAczWKKMZlFjch videos with polyps.zip
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1gSJiUhZJnpkELXoCUgNsPtYnUCS7qqyP
From (redirected): https://drive.google.com/uc?id=1gSJiUhZJnpkELXoCUgNsPtYnUCS7qqyP&confirm=t&uuid=bb68003b-60a8-4102-af8d-c4fb8579e1be
To: /content/drive/MyDrive/data_lake/01_bronze_medical/ldpolypvideo/Test.rar
100% 1.15G/1.15G [00:26<00:00, 43.0MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Y-fS9vMGvrEQZS1rKDnCxSkvrQz1lyXX
From (redirected): https://drive.google.com/uc?id=1Y-fS9vMGvrEQZS1rKDnCxSkvrQz1lyXX&confirm=t&uuid=a887c003-825d-490e-a394-4ba939994675
To: /content/drive/MyDrive/data_lake/01_bron

In [6]:
# Cell 6: Final verification of ALL polyp datasets
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/data_lake')
all_datasets = [
    '01_bronze_medical/kvasir-seg',
    '01_bronze_medical/cvc-clinicdb',
    '01_bronze_medical/cvc-colondb',
    '01_bronze_medical/bkai-igh',
    '01_bronze_medical/etis-larib',
    '01_bronze_medical/polypdb',
    '01_bronze_medical/ldpolypvideo',
    '01_bronze_medical/polypgen',
]

print('\n' + '='*60)
print('FINAL VERIFICATION — ALL POLYP DATASETS')
print('='*60)
total_ok = 0
for ds in all_datasets:
    path = DRIVE / ds
    if path.exists():
        files = list(path.rglob('*'))
        file_count = sum(1 for f in files if f.is_file())
        total_size = sum(f.stat().st_size for f in files if f.is_file())
        size_mb = total_size / 1024 / 1024
        ok = file_count > 0 and size_mb > 1
        icon = '\u2705' if ok else '\u26a0\ufe0f'
        total_ok += 1 if ok else 0
        print(f'{icon} {ds} — {file_count} files, {size_mb:.1f} MB')
    else:
        print(f'\u274c {ds} — NOT FOUND')

print(f'\nTotal: {total_ok}/{len(all_datasets)} datasets ready')


FINAL VERIFICATION — ALL POLYP DATASETS
✅ 01_bronze_medical/kvasir-seg — 4004 files, 152.3 MB
✅ 01_bronze_medical/cvc-clinicdb — 2450 files, 309.5 MB
✅ 01_bronze_medical/cvc-colondb — 1520 files, 182.5 MB
❌ 01_bronze_medical/bkai-igh — NOT FOUND
✅ 01_bronze_medical/etis-larib — 392 files, 176.7 MB
❌ 01_bronze_medical/polypdb — NOT FOUND
✅ 01_bronze_medical/ldpolypvideo — 64 files, 25409.4 MB
✅ 01_bronze_medical/polypgen — 17 files, 7.6 MB

Total: 6/8 datasets ready


In [7]:
# Cell 7: Refresh MANIFEST
from shared_config.manifest import generate_manifest
generate_manifest()
print('\n\u2705 MANIFEST.json updated')

=== Generating MANIFEST.json ===
  01_bronze_audio: 5 datasets
  01_bronze_detection: 3 datasets
  01_bronze_education: 11 datasets
  01_bronze_generative: 1 datasets
  01_bronze_medical: 31 datasets
  01_bronze_nlp: 18 datasets
  01_bronze_tabular: 19 datasets
  01_bronze_timeseries: 5 datasets
  01_bronze_video: 0 datasets
  01_bronze_vision: 45 datasets

✅ MANIFEST.json written to /content/drive/MyDrive/data_lake/MANIFEST.json

📊 Summary:
   Total datasets: 138
   Total files: 804,242
   Total size: 169.59 GB
   Empty datasets: 0

✅ MANIFEST.json updated
